# Break Time 탐지 — 시간 컬럼 + break_time 추가

- **참고**: `eda.ipynb` Cell 29 + `basic_eda.ipynb` Cell 5 로직 재사용
- **입력**: `lectures_kss.csv`, `lectures_origin.csv`
- **출력**: `lectures_kss_with_break.csv`, `lectures_origin_with_break.csv`

| 추가 컬럼 | 설명 |
|---|---|
| `sec_raw` | timestamp(HH:MM:SS) → 절대 초 |
| `sec_fixed` | 오전→오후 역전 보정 후 절대 초 |
| `elapsed_sec` | 강의 시작 기준 경과 초 |
| `duration_sec` | 다음 발화까지 시간 차이(초), 마지막 행 = 0 |
| `break_time` | 1 = 쉬는 시간 직전 경계 발화, 0 = 나머지 |

In [10]:
import re
import json
from pathlib import Path

import pandas as pd

DATA = Path('../data/processed')

## 1. 헬퍼 함수 (basic_eda.ipynb + eda.ipynb 로직)

In [11]:
# ── 시간 변환 (basic_eda.ipynb Cell 5) ────────────────────────────────────────

def ts_to_seconds(ts: str) -> int:
    """HH:MM:SS → 절대 초"""
    h, m, s = map(int, ts.split(':'))
    return h * 3600 + m * 60 + s


def fix_am_pm_timestamps(seconds_list: list, min_backward_jump_sec: int = 300) -> list:
    """
    이전 timestamp보다 min_backward_jump_sec 이상 크게 감소하면
    오전→오후 전환으로 간주해 +12시간 보정.
    작은 역전은 STT/정렬 오류로 간주.
    """
    if not seconds_list:
        return []
    offset = 0
    fixed  = []
    prev   = seconds_list[0]
    for sec in seconds_list:
        if sec < prev and (prev - sec) >= min_backward_jump_sec:
            offset += 12 * 3600
        fixed.append(sec + offset)
        prev = sec
    return fixed


# ── Break 탐지 설정 (eda.ipynb Cell 29) ──────────────────────────────────────

BREAK_KEYWORDS   = ['쉬는', '쉬었', '쉬고', '쉬도록', '쉬세', '쉴', '쉬겠', '쉬자', '잠깐 쉬', '잠시 쉬', '식사']
MIN_GAP_SEC      = 600    # 10분: 이 이상 gap일 때만 break 후보
GAP_FALLBACK_SEC = 3600   # 1시간: 키워드 없어도 무조건 break
KEYWORD_WINDOW   = 2      # 경계 발화 + 직전 N개 발화 윈도우

def has_break_keyword(texts: list) -> bool:
    return any(kw in t for t in texts for kw in BREAK_KEYWORDS)

def detect_breaks(g: pd.DataFrame) -> pd.Series:
    """그룹(강의 1일치) 내에서 각 행이 break 경계 발화인지 0/1로 반환."""
    result = pd.Series(0, index=g.index)
    for idx, row in g.iterrows():
        if row['duration_sec'] < MIN_GAP_SEC:
            continue
        if row['duration_sec'] >= GAP_FALLBACK_SEC:
            result[idx] = 1
            continue
        window_start = max(g.index[0], idx - KEYWORD_WINDOW)
        window_texts = g.loc[window_start:idx, 'text_raw'].tolist()
        if has_break_keyword(window_texts):
            result[idx] = 1
    return result


# ── 전체 처리 파이프라인 ────────────────────────────────────────────────────────

def enrich(df: pd.DataFrame) -> pd.DataFrame:
    """
    sec_raw / sec_fixed / elapsed_sec / duration_sec / break_time 컬럼 추가.
    lecture_id + date 기준 그룹 내에서 계산.
    """
    df = df.copy()

    # 1) sec_raw: timestamp → 절대 초
    df['sec_raw'] = df['timestamp'].apply(ts_to_seconds)

    # 2) sec_fixed: 그룹 내 오전→오후 역전 보정
    def _apply_fix(g):
        fixed = fix_am_pm_timestamps(g['sec_raw'].tolist())
        return pd.Series(fixed, index=g.index)

    df['sec_fixed'] = (
        df.groupby(['lecture_id', 'date'], group_keys=False)
          .apply(_apply_fix)
    )

    # 3) elapsed_sec: 강의 시작(sec_fixed 최솟값) 기준 경과 초
    df['elapsed_sec'] = (
        df.groupby(['lecture_id', 'date'])['sec_fixed']
          .transform(lambda s: s - s.min())
    ).astype(int)

    # 4) duration_sec: 다음 발화까지 시간 차이, 마지막 행 = 0
    df['duration_sec'] = (
        df.groupby(['lecture_id', 'date'])['elapsed_sec']
          .transform(lambda s: s.diff().shift(-1).fillna(0))
    ).astype(int)

    # 5) break_time: 쉬는 시간 경계 발화 여부
    df['break_time'] = (
        df.groupby(['lecture_id', 'date'], group_keys=False)
          .apply(detect_breaks)
    ).astype(int)

    return df


print('함수 정의 완료')

함수 정의 완료


## 2. lectures_kss.csv 처리

In [12]:
kss = pd.read_csv(DATA / 'lectures_kss.csv')
print(f'원본: {len(kss):,}행  |  컬럼: {kss.columns.tolist()}')

kss_out = enrich(kss)

print(f'\n추가된 컬럼: {[c for c in kss_out.columns if c not in kss.columns]}')
print(f'\n[kss] break_time 분포:')
print(kss_out['break_time'].value_counts().sort_index().to_string())
print(f'\n경계 발화 수: {kss_out["break_time"].sum()}건 '
      f'(강의당 평균 {kss_out["break_time"].sum() / kss_out["lecture_id"].nunique():.1f}개)')

원본: 24,238행  |  컬럼: ['lecture_id', 'date', 'timestamp', 'speaker_id', 'text_raw']


/var/folders/4w/4hfq6k852wg71r8sw418pl_40000gn/T/ipykernel_77467/4103116699.py:73: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_apply_fix)



추가된 컬럼: ['sec_raw', 'sec_fixed', 'elapsed_sec', 'duration_sec', 'break_time']

[kss] break_time 분포:
break_time
0    24193
1       45

경계 발화 수: 45건 (강의당 평균 45.0개)


/var/folders/4w/4hfq6k852wg71r8sw418pl_40000gn/T/ipykernel_77467/4103116699.py:91: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(detect_breaks)


In [13]:
# 시간 컬럼 sanity check
print('[kss] sec_raw / sec_fixed / elapsed_sec / duration_sec 샘플 (첫 5행):')
display(
    kss_out[['lecture_id', 'date', 'timestamp', 'sec_raw', 'sec_fixed', 'elapsed_sec', 'duration_sec', 'break_time']]
    .head(5)
)

# 오전→오후 보정이 발생한 강의 확인
fixed_diff = kss_out['sec_fixed'] - kss_out['sec_raw']
corrected = kss_out[fixed_diff > 0][['lecture_id', 'date', 'timestamp', 'sec_raw', 'sec_fixed']]
print(f'\n오전→오후 보정된 발화 수: {len(corrected)}건')
if len(corrected) > 0:
    display(corrected.drop_duplicates('lecture_id').head(5))

[kss] sec_raw / sec_fixed / elapsed_sec / duration_sec 샘플 (첫 5행):


,lecture_id,date,timestamp,sec_raw,sec_fixed,elapsed_sec,duration_sec,break_time
0,kdt-backendj-21th,2026-02-02,09:11:17,33077,33077,0,0,0
1,kdt-backendj-21th,2026-02-02,09:11:17,33077,33077,0,0,0
2,kdt-backendj-21th,2026-02-02,09:11:17,33077,33077,0,0,0
3,kdt-backendj-21th,2026-02-02,09:11:17,33077,33077,0,1,0
4,kdt-backendj-21th,2026-02-02,09:11:18,33078,33078,1,0,0



오전→오후 보정된 발화 수: 12115건


,lecture_id,date,timestamp,sec_raw,sec_fixed
590,kdt-backendj-21th,2026-02-02,01:00:30,3630,46830


In [15]:
# break_time == 1 샘플 확인
print('[kss] break 경계 발화 샘플:')
display(
    kss_out[kss_out['break_time'] == 1]
    [['lecture_id', 'date', 'timestamp', 'elapsed_sec', 'duration_sec', 'text_raw']]
    .head(50)
)

[kss] break 경계 발화 샘플:


,lecture_id,date,timestamp,elapsed_sec,duration_sec,text_raw
585,kdt-backendj-21th,2026-02-02,10:43:08,5511,3903,11시 그렇지
589,kdt-backendj-21th,2026-02-02,11:48:26,9429,4324,되셨어요.
1305,kdt-backendj-21th,2026-02-02,02:50:25,20348,1235,3시 10분에 오겠습니다.
2199,kdt-backendj-21th,2026-02-03,10:52:35,6091,1258,11시 10분에 오겠습니다.
2537,kdt-backendj-21th,2026-02-03,11:58:48,10064,4688,식사 맛있게 하고 식사하셨어요.
3666,kdt-backendj-21th,2026-02-03,04:21:00,25796,5360,2시가 되 알겠어요.
4241,kdt-backendj-21th,2026-02-04,10:28:42,4700,1969,일단 여기까지 하고 잠시 쉴게요 오늘은 저희 50분에 쉬겠습니다니다.
4664,kdt-backendj-21th,2026-02-04,12:00:54,10232,4829,식사 맛있게 하고 오십시오.
5232,kdt-backendj-21th,2026-02-04,02:49:30,20348,1246,지금 50분이라서 쉬고 3시 10분에 오도록 하겠습니다.
5655,kdt-backendj-21th,2026-02-04,04:18:50,25708,5484,얘가 구조만 삭제하는 거니까 꼭 핸드에서 해보고 여기서 해보기 저희 20분 쉬었다가...


In [16]:
out_path = DATA / 'lectures_kss_with_break.csv'
kss_out.to_csv(out_path, index=False)
print(f'저장 완료: {out_path}  ({len(kss_out):,}행  |  컬럼: {kss_out.columns.tolist()})')

저장 완료: ../data/processed/lectures_kss_with_break.csv  (24,238행  |  컬럼: ['lecture_id', 'date', 'timestamp', 'speaker_id', 'text_raw', 'sec_raw', 'sec_fixed', 'elapsed_sec', 'duration_sec', 'break_time'])


## 3. lectures_origin.csv 처리

In [17]:
origin = pd.read_csv(DATA / 'lectures_origin.csv')
print(f'원본: {len(origin):,}행  |  컬럼: {origin.columns.tolist()}')

origin_out = enrich(origin)

print(f'\n[origin] break_time 분포:')
print(origin_out['break_time'].value_counts().sort_index().to_string())
print(f'\n경계 발화 수: {origin_out["break_time"].sum()}건 '
      f'(강의당 평균 {origin_out["break_time"].sum() / origin_out["lecture_id"].nunique():.1f}개)')

원본: 22,756행  |  컬럼: ['lecture_id', 'date', 'timestamp', 'speaker_id', 'text_raw']


/var/folders/4w/4hfq6k852wg71r8sw418pl_40000gn/T/ipykernel_77467/4103116699.py:73: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_apply_fix)



[origin] break_time 분포:
break_time
0    22711
1       45

경계 발화 수: 45건 (강의당 평균 45.0개)


/var/folders/4w/4hfq6k852wg71r8sw418pl_40000gn/T/ipykernel_77467/4103116699.py:91: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(detect_breaks)


In [18]:
print('[origin] break 경계 발화 샘플:')
display(
    origin_out[origin_out['break_time'] == 1]
    [['lecture_id', 'date', 'timestamp', 'elapsed_sec', 'duration_sec', 'text_raw']]
    .head(10)
)

[origin] break 경계 발화 샘플:


,lecture_id,date,timestamp,elapsed_sec,duration_sec,text_raw
543,kdt-backendj-21th,2026-02-02,10:43:08,5511,3903,11시 43분이에요. 11시
547,kdt-backendj-21th,2026-02-02,11:48:26,9429,4324,되셨어요.
1184,kdt-backendj-21th,2026-02-02,02:50:25,20348,1235,3시 10분에 오겠습니다.
2069,kdt-backendj-21th,2026-02-03,10:52:36,6092,1257,10분에 오겠습니다.
2371,kdt-backendj-21th,2026-02-03,11:58:48,10064,4688,식사 맛있게 하고 식사하셨어요.
3339,kdt-backendj-21th,2026-02-03,04:21:00,25796,5360,2시가 되 알겠어요.
3874,kdt-backendj-21th,2026-02-04,10:28:42,4700,1969,일단 여기까지 하고 잠시 쉴게요 오늘은 저희 50분에 쉬겠습니다니다.
4244,kdt-backendj-21th,2026-02-04,12:00:54,10232,4829,"오늘은요, 지금 12시고 제가 또 10분 여러분들 저화해서 1시 20분에 시작하도록..."
4745,kdt-backendj-21th,2026-02-04,02:49:37,20355,1239,50분이라서 쉬고 3시 10분에 오도록 하겠습니다.
5113,kdt-backendj-21th,2026-02-04,04:19:07,25725,5467,삭제하는 거니까 꼭 핸드에서 해보고 여기서 해보기 저희 20분 쉬었다가 20분 쉬세요.


In [19]:
out_path = DATA / 'lectures_origin_with_break.csv'
origin_out.to_csv(out_path, index=False)
print(f'저장 완료: {out_path}  ({len(origin_out):,}행  |  컬럼: {origin_out.columns.tolist()})')

저장 완료: ../data/processed/lectures_origin_with_break.csv  (22,756행  |  컬럼: ['lecture_id', 'date', 'timestamp', 'speaker_id', 'text_raw', 'sec_raw', 'sec_fixed', 'elapsed_sec', 'duration_sec', 'break_time'])


## 4. 교차 검증 — kss vs origin break 시점 일치 확인

In [20]:
kss_set = set(
    kss_out[kss_out['break_time'] == 1]
    .set_index(['lecture_id', 'date', 'timestamp']).index
)
origin_set = set(
    origin_out[origin_out['break_time'] == 1]
    .set_index(['lecture_id', 'date', 'timestamp']).index
)

print(f'kss    break 수: {len(kss_set)}')
print(f'origin break 수: {len(origin_set)}')
print(f'공통  timestamp: {len(kss_set & origin_set)}')
print(f'kss에만 있는 것: {len(kss_set - origin_set)}')
print(f'origin에만 있는 것: {len(origin_set - kss_set)}')

if kss_set - origin_set:
    print('\n[kss에만] 샘플:')
    for t in list(kss_set - origin_set)[:5]:
        print(' ', t)

if origin_set - kss_set:
    print('\n[origin에만] 샘플:')
    for t in list(origin_set - kss_set)[:5]:
        print(' ', t)

kss    break 수: 45
origin break 수: 45
공통  timestamp: 40
kss에만 있는 것: 5
origin에만 있는 것: 5

[kss에만] 샘플:
  ('kdt-backendj-21th', '2026-02-03', '10:52:35')
  ('kdt-backendj-21th', '2026-02-24', '11:56:23')
  ('kdt-backendj-21th', '2026-02-04', '02:49:30')
  ('kdt-backendj-21th', '2026-02-04', '04:18:50')
  ('kdt-backendj-21th', '2026-02-26', '12:04:00')

[origin에만] 샘플:
  ('kdt-backendj-21th', '2026-02-24', '11:56:41')
  ('kdt-backendj-21th', '2026-02-04', '02:49:37')
  ('kdt-backendj-21th', '2026-02-26', '12:04:01')
  ('kdt-backendj-21th', '2026-02-04', '04:19:07')
  ('kdt-backendj-21th', '2026-02-03', '10:52:36')
